In [1]:
from dataGeneration import run_scr_model
from minMaxLHSampler import maximin_lhs

import numpy as np

mat_path = "./data/NEDC_lite.mat"

In [2]:
mean_k_ads = 4.32
mean_k_des = 4.83e04
mean_k_std = 3.59e09
mean_k_fst = 8.80e09
mean_k_slw = 3.56e11
mean_k_oxi = 6.99e05

one_std_k_ads = 1.8
one_std_k_des = 3.22e04
one_std_k_std = 7.18e07
one_std_k_fst = 1.76e08
one_std_k_slw = 7.13e09
one_std_k_oxi = 2.33e05

std_k_ads = mean_k_ads - one_std_k_ads
std_k_des = mean_k_des - one_std_k_des
std_k_std = mean_k_std - one_std_k_std
std_k_fst = mean_k_fst - one_std_k_fst
std_k_slw = mean_k_slw - one_std_k_slw
std_k_oxi = mean_k_oxi - one_std_k_oxi

In [3]:
up_k_ads = mean_k_ads + std_k_ads
up_k_des = mean_k_des + std_k_des
up_k_std = mean_k_std + std_k_std
up_k_fst = mean_k_fst + std_k_fst
up_k_slw = mean_k_slw + std_k_slw
up_k_oxi = mean_k_oxi + std_k_oxi

low_k_ads = mean_k_ads - std_k_ads
low_k_des = mean_k_des - std_k_des
low_k_std = mean_k_std - std_k_std
low_k_fst = mean_k_fst - std_k_fst
low_k_slw = mean_k_slw - std_k_slw
low_k_oxi = mean_k_oxi - std_k_oxi

lower_bounds = [low_k_ads, low_k_des, low_k_std, low_k_fst, low_k_slw, low_k_oxi]
upper_bounds = [up_k_ads, up_k_des, up_k_std, up_k_fst, up_k_slw, up_k_oxi]

# ads = 0
# des = 1
# std = 2
# fst = 3
# slw = 4
# oxi = 5


In [4]:
best_sample = maximin_lhs(lower_bounds=lower_bounds, upper_bounds=upper_bounds, n_samples=1000, n_candidates=10000)

In [5]:
np.save("lhs_sampling.npy", best_sample)
print(best_sample.shape)

(1000, 6)


In [ ]:
from tqdm import tqdm

for _ in tqdm(range(len(best_sample))):
    k_ads = best_sample[_][0]
    k_des = best_sample[_][1]
    k_std = best_sample[_][2]
    k_fst = best_sample[_][3]
    k_slw = best_sample[_][4]
    k_oxi = best_sample[_][5]

    time_s, Model_sensor_ppm = run_scr_model(
                                mat_path = mat_path,
                                k_std_0=k_std,
                                k_fst_0=k_fst,
                                k_slw_0=k_slw,
                                k_ads_0=k_ads,
                                k_des_0=k_des,
                                k_nh3ox_0=k_oxi,
                            )
    
    if _ == 0:
        data = np.expand_dims(Model_sensor_ppm, axis = 0)
    else:
        data = np.concatenate((data, np.expand_dims(Model_sensor_ppm, axis = 0)))

    if ((_+1) % 10 == 0) and (_ != 0):
        print(f"Saved data for {_+1} samples.")
        np.save("lhs_data.npy", data)
    
np.save("lhs_data.npy", data)

  1%|          | 10/1000 [03:32<5:48:31, 21.12s/it]

Saved data for 9 samples.


  1%|▏         | 13/1000 [04:46<6:02:16, 22.02s/it]


KeyboardInterrupt: 